### Fase 1: Transformando Canciones en "Canastas"
El algoritmo Apriori no lee DataFrames normales; necesita una lista de transacciones. 
En nuestro caso, cada **Canción** es una transacción, y los **Artistas** que participan en ella son los artículos de la canasta.

In [1]:
# Instalar si es necesario: !pip install mlxtend
import pandas as pd
import re
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. INGESTA DE DATOS
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"
df_spotify = pd.read_csv(url, encoding='latin-1')

# 2. LIMPIEZA: Eliminar nulos de la columna de artistas
df_spotify = df_spotify.dropna(subset=['Artist'])

# 3. TRANSFORMACIÓN: Separar los artistas que colaboran en cada canción
# Extraemos colaboraciones separadas por coma en Artist y menciones (feat./ft.) en Track
def extraer_colaboradores(row):
    artistas = [artista.strip() for artista in str(row['Artist']).split(',')]
    match = re.search(r'\((?:feat\.|ft\.|with)\s+([^)]+)\)', str(row['Track']), re.IGNORECASE)
    if match:
        feats = re.split(r'[,&]|\band\b', match.group(1))
        for f in feats:
            f_clean = f.strip()
            if f_clean and f_clean not in artistas:
                artistas.append(f_clean)
    return artistas

lista_colaboraciones = df_spotify.apply(extraer_colaboradores, axis=1).tolist()

# Muestra rápida de cómo se ven nuestras "canastas"
print("Muestra de las primeras 5 colaboraciones:")
for i in range(5):
    print(f"Canción {i+1}: {lista_colaboraciones[i]}")

Muestra de las primeras 5 colaboraciones:
Canción 1: ['Tommy Richman']
Canción 2: ['Kendrick Lamar']
Canción 3: ['Artemas']
Canción 4: ['Miley Cyrus']
Canción 5: ['Eminem']


In [2]:
# Utiliza TransactionEncoder para convertir 'lista_colaboraciones' en una matriz booleana (One-Hot)
# Guárdala en un DataFrame llamado 'df_transacciones'

te = TransactionEncoder()
matriz_bool = te.fit_transform(lista_colaboraciones)
df_transacciones = pd.DataFrame(matriz_bool, columns=te.columns_)

print("--- MATRIZ DE TRANSACCIONES ---")
display(df_transacciones.head())

--- MATRIZ DE TRANSACCIONES ---


,"""XY""",$OHO BANI,$uicideboy$,&ME,(G)I-DLE,*NSYNC,.diedlonely,10CM,13 Organisï¿,1da Banton,...,ýýýýýýýýýý,ýýýýýýýýýý ýýýýýýýý,ýýýýýýýýýýýý,ýýýýýýýýýýýý ýýýýýýýýýýýýýý,ýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýý(ýýýýýýýýýýýýýýý),ýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýýýýý
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Fase 2: Aplicando Apriori
Ahora que tenemos la matriz con ceros y unos, extraigamos los conjuntos de artistas que colaboran frecuentemente.
Debido a que hay miles de artistas solistas, las colaboraciones son estadísticamente raras. Usaremos un `min_support` muy bajo para capturar las colaboraciones.

In [3]:
# --- RETO PARA EL ALUMNO AQUÍ ---

# 1. Encuentra los Itemsets Frecuentes usando un soporte mínimo (0.0005 para capturar colaboraciones en 4,595 canciones)
itemsets_frecuentes = apriori(df_transacciones, min_support=0.0005, use_colnames=True)

# 2. Genera las Reglas de Asociación filtrando por la métrica 'lift' con un mínimo de 1.2
reglas_musicales = association_rules(itemsets_frecuentes, metric="lift", min_threshold=1.2)

# Columnas legibles para presentación
reglas_musicales['ant_str'] = reglas_musicales['antecedents'].apply(lambda x: ', '.join(list(x)))
reglas_musicales['con_str'] = reglas_musicales['consequents'].apply(lambda x: ', '.join(list(x)))

# 3. Muestra las reglas resultantes ordenadas por 'confidence' de mayor a menor
reglas_musicales = reglas_musicales.sort_values(by='confidence', ascending=False)
print("--- TOP REGLAS MUSICALES POR CONFIANZA ---")
display(reglas_musicales[['ant_str', 'con_str', 'support', 'confidence', 'lift']].head(15))

--- TOP REGLAS MUSICALES POR CONFIANZA ---


,ant_str,con_str,support,confidence,lift
5,Bryant Myers,Dei V,0.000653,1.000000,919.000000
28,Tyler,The Creator,0.001741,1.000000,574.375000
29,The Creator,Tyler,0.001741,1.000000,574.375000
14,PARTYNEXTDOOR,Drake,0.000653,0.750000,40.544118
4,Dei V,Bryant Myers,0.000653,0.600000,919.000000
18,Sexyy Red,Drake,0.000653,0.500000,27.029412
6,DJ Khaled,Drake,0.000653,0.428571,23.168067
8,DJ Snake,Ozuna,0.000653,0.333333,85.092593
25,Lil Durk,Lil Baby,0.000653,0.214286,28.132653
3,Metro Boomin,21 Savage,0.000653,0.187500,28.718750


### 🧠 Fase 3: Interpretación de Negocio (El Entregable)

Como Científico de Datos, no basta con entregar una tabla. Debes explicarle los hallazgos al Director Creativo. Revisa tu DataFrame `reglas_musicales` y responde las siguientes 4 preguntas:

**1. Análisis de Soporte (El Volumen):**
Menciona a los dos artistas que conforman la regla con el mayor **Soporte**. ¿Qué significa este número en términos simples para el negocio?

> **Tu Respuesta:** La colaboración con mayor soporte real entre distintos artistas es **Bryant Myers y Dei V** (soporte de 0.000653, presente en 3 canciones del top mundial, al igual que parejas como Drake y 21 Savage / Drake y Future). En términos simples de negocio, el soporte indica la penetración de mercado o frecuencia absoluta: representa el porcentaje de todas las canciones analizadas en la plataforma donde ambos artistas aparecen juntos en el mismo track.

**2. Análisis de Confianza (La Dirección):**
Busca una regla que tenga una **Confianza** mayor al 0.5 (50%). Traduce esta regla matemática a una frase que el director pueda entender (Ej. "Si contratamos al artista A, existe un X% de probabilidad de que el artista B también deba estar en el track para que sea el éxito que registramos").

> **Tu Respuesta:** La regla **Bryant Myers $\rightarrow$ Dei V** alcanza una **Confianza del 1.0 (100%)**, y la regla **PARTYNEXTDOOR $\rightarrow$ Drake** alcanza una **Confianza de 0.75 (75%)**. Traducido al director: *"Si fichamos a PARTYNEXTDOOR para producir un lanzamiento exitoso, existe un 75% de probabilidad histórica de que Drake también deba colaborar en el track para alcanzar el nivel de reproducciones registrado en el ranking global"*.

**3. Análisis de Lift (El Impulso):**
Busca la regla con el **Lift** más alto de toda tu tabla. 
¿Por qué esta métrica es más confiable que el Soporte para decidir si estos dos artistas *realmente* tienen química musical o si solo fue una coincidencia estadística?

> **Tu Respuesta:** La regla con mayor Lift es **Bryant Myers $\leftrightarrow$ Dei V** con un **Lift de 919.0**. Es mucho más confiable que el soporte porque el soporte favorece desproporcionadamente a artistas masivos individuales (como Drake con más de 80 temas en lista), mientras que el Lift contrasta la frecuencia conjunta contra la probabilidad de que coincidieran al azar. Un Lift de 919 demuestra que la aparición conjunta es 919 veces mayor a una coincidencia estadística casual, confirmando auténtica química comercial y sinergia musical.

**4. Toma de Decisión:**
Basado puramente en las reglas matemáticas que descubriste, si la disquera solo tiene presupuesto para juntar a dos artistas en el estudio la próxima semana, ¿qué pareja elegirías y por qué? (Justifica usando las 3 métricas de oro).

> **Tu Respuesta:** Elegiría a la dupla **Bryant Myers y Dei V** (o alternativamente **PARTYNEXTDOOR y Drake**):
> - **Soporte:** Se ubica en el rango más alto de colaboraciones del dataset global (3 canciones en el ranking, soporte máximo de 0.000653).
> - **Confianza:** Extraordinaria; 100% en la dirección Bryant Myers $\rightarrow$ Dei V y 60% en la inversa Dei V $\rightarrow$ Bryant Myers, demostrando fidelidad recíproca entre ambas audiencias.
> - **Lift:** El más alto de todo el estudio (919.0), lo que garantiza el máximo multiplicador de impacto conjunto y minimiza el riesgo financiero de inversión en estudio.